# Star Formation

This notebook shows how to render **grid-based snapshots** from Athena++:

```text
AMR snapshot -> export_volume_vdb_sequence() -> .vdb file -> setup_animation() in Blender
```

Use this when the source data is already volume data, for example Athena++ or another AMR code.


## 1. Export VDB frames

Run this section in a Python 3.11 environment with `yt` and `openvdb` available.


In [ ]:
import os

from AstroVis.backend import export_volume_vdb_sequence

input_dir = r"Data"
output_dir = r"Output"
field_name = "density"
num_snapshot = 5  

export_info = export_volume_vdb_sequence(input_dir, output_dir, field=field_name, num_snapshot=num_snapshot, preview_every = 1)
preview_range = export_info["field_range"]


In [ ]:
# Check ds.field_list first when you want to confirm the yt particle type and field name.

import yt

snapshots = sorted([f for f in os.listdir(input_dir) if f.endswith(".athdf")])
ds = yt.load(os.path.join(input_dir, snapshots[0]))
print(ds.field_list)

## 2. Import and render in Blender

Run this section **inside Blender's Python environment**.

The volume material already emits light from the data values, so this example keeps the scene simple and does **not** add extra lights.


In [ ]:
import bpy

import sys
sys.path.append("../..") #Set the path to the AstroVis package if it's not in the default Python path

from AstroVis.api import *

vdb_folder = r"Path/to/your/vdb/folder"  # Replace with the actual path to your VDB folder

scene = SceneManager()
scene.clear_scene(use_empty=False)

#vdb_folder = r"D:\M1AD1_multi_vdb"
species_name = "Gas"

# Keep this range fixed across the whole animation so every frame uses the same visual scale.
material = create_field_volume_material(
    species_name,
    field_min=-2,
    field_max=5,
    apply=False,
)[species_name]

setup_animation(data_path=vdb_folder, object=species_name, material=material)
scene.set_camera(location=(8, -8, 5), look_at=(0, 0, 0))

## Notes

- `ds.field_list` is the first thing to check when you are unsure about the yt field type or the field name.
- `export_volume_sequence()` is the simple one-pass helper: it exports every frame, saves preview images every 10th frame by default, and returns the final global range for Blender.
- Keep the same `field_min` and `field_max` across all frames so the Blender material uses one consistent scale.
- The examples here intentionally use only the minimum useful arguments. For optional controls, see `Documentation/Backend/Volume_Data.md`, `Documentation/Blender/Animation_setup.md`, and `Documentation/Blender/Node_and_shading.md`.
- `setup_volume_animation()` is the lower-level alternative when you want to import the VDB sequence directly instead of using `setup_animation()`.
